In [2]:
from keras import layers
from keras import Input
from keras.models import Model

import numpy as np
import tqdm
import keras    
import tensorflow as tf
import os
import csv
import pathlib
import unicode

#from tensorflow.python.keras.preprocessing.image import ImageDataGenerator

In [3]:
#PRE DEFINE

DATA_SIZE = 30000
VALID_DATA_SIZE = DATA_SIZE / 5

ORG_TRAIN_CSV_PATH = ""
SHUF_TRAIN_CSV_PATH = ""

ORG_VALID_CSV_PATH = ""
SHUF_TRAIN_CSV_PATH = ""

In [4]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

print(tf.__version__)
from tensorflow.python.client import device_lib
device_lib.list_local_devices()

2.16.1


2024-04-09 08:50:31.434867: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-09 08:50:31.667909: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-09 08:50:31.668016: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-09 08:50:31.837307: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-09 08:50:31.837369: I external/local_xla/xla/stream_executor

[name: "/device:CPU:0"
 device_type: "CPU"
 memory_limit: 268435456
 locality {
 }
 incarnation: 17976852520981599646
 xla_global_id: -1,
 name: "/device:GPU:0"
 device_type: "GPU"
 memory_limit: 2355888128
 locality {
   bus_id: 1
   links {
   }
 }
 incarnation: 10736720312388778306
 physical_device_desc: "device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5"
 xla_global_id: 416903419]

In [5]:
ja2label = {'ㄱ':0, 'ㄲ':1, 'ㄴ':2, 'ㄷ':3, 'ㄸ':4, 'ㄹ':5, 'ㅁ':6, 'ㅂ':7, 'ㅃ':8,
'ㅅ':9, 'ㅆ':10, 'ㅇ':11,  'ㅈ':12, 'ㅉ':13, 'ㅊ':14, 'ㅋ':15, 'ㅌ':16,  'ㅍ':17, 'ㅎ':18}

mo2label = {'ㅏ':0, 'ㅐ':1, 'ㅑ':2, 'ㅒ':3, 'ㅓ':4, 'ㅔ':5, 'ㅕ':6, 'ㅖ':7, 'ㅗ':8, 'ㅘ':9, 
'ㅙ':10, 'ㅚ':11, 'ㅛ':12, 'ㅜ':13, 'ㅝ':14, 'ㅞ':15, 'ㅟ':16, 'ㅠ':17, 'ㅡ':18, 'ㅢ':19, 'ㅣ':20}

ba2label = {None:0, 'ㄱ':1, 'ㄲ':2, 'ㄳ':3, 'ㄴ':4, 'ㄵ':5, 'ㄶ':6, 'ㄷ':7, 'ㄹ':8, 'ㄺ':9,
'ㄻ':10, 'ㄼ':11, 'ㄽ':12, 'ㄾ':13, 'ㄿ':14, 'ㅀ':15, 'ㅁ':16, 'ㅂ':17, 'ㅄ':18, 'ㅅ':19,
'ㅆ':20, 'ㅇ':21, 'ㅈ':22, 'ㅊ':23, 'ㅋ':24, 'ㅌ':25, 'ㅍ':26, 'ㅎ':27}

label2ja = {0: 'ㄱ', 1: 'ㄲ', 2: 'ㄴ', 3: 'ㄷ', 4: 'ㄸ', 5: 'ㄹ',
            6: 'ㅁ', 7: 'ㅂ', 8: 'ㅃ', 9: 'ㅅ', 10: 'ㅆ', 11: 'ㅇ',
            12: 'ㅈ', 13: 'ㅉ', 14: 'ㅊ', 15: 'ㅋ', 16: 'ㅌ', 17: 'ㅍ', 18: 'ㅎ'}

label2mo = {0: 'ㅏ', 1: 'ㅐ', 2: 'ㅑ', 3: 'ㅒ', 4: 'ㅓ', 5: 'ㅔ',
            6: 'ㅕ', 7: 'ㅖ', 8: 'ㅗ', 9: 'ㅘ', 10: 'ㅙ', 11: 'ㅚ',
            12: 'ㅛ', 13: 'ㅜ', 14: 'ㅝ', 15: 'ㅞ', 16: 'ㅟ', 17: 'ㅠ',
            18: 'ㅡ', 19: 'ㅢ', 20: 'ㅣ'}

label2ba = {0: None, 1: 'ㄱ', 2: 'ㄲ', 3: 'ㄳ', 4: 'ㄴ', 5: 'ㄵ',
            6: 'ㄶ', 7: 'ㄷ', 8: 'ㄹ', 9: 'ㄺ', 10: 'ㄻ', 11: 'ㄼ',
            12: 'ㄽ', 13: 'ㄾ', 14: 'ㄿ', 15: 'ㅀ', 16: 'ㅁ', 17: 'ㅂ',
            18: 'ㅄ', 19: 'ㅅ', 20: 'ㅆ', 21: 'ㅇ', 22: 'ㅈ', 23: 'ㅊ',
            24: 'ㅋ', 25: 'ㅌ', 26: 'ㅍ', 27: 'ㅎ'}

In [6]:

def getSpecificExtensionFiles(path, extension):
    out = []
    for (path, dir, files) in os.walk(path):
        for filename in files:
            ext = os.path.splitext(filename)[-1]
            if ext == extension:
                #print("%s/%s" % (path, filename))
                out.append(path + "/" + filename)
    return out

In [7]:
import matplotlib.pyplot as plt
import random

# draw_text = '람'
# font = "/root/Data/font/clova-all/가람연꽃/나눔손글씨_가람연꽃.ttf"

# fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")

from PIL import Image,ImageDraw,ImageFont

def CreateFontImage(str, fontPath):
    font = ImageFont.truetype(fontPath, 28, encoding = 'utf-8')
    left, top, right, bottom = font.getbbox(str)
    width = right - left
    height = bottom - top
    
    canvas = Image.new('RGB', (width + 10, height + 14), "white")
    draw = ImageDraw.Draw(canvas)
    draw.text((3,3), str, 'black', font)
    
    #print(canvas)
    img = tf.image.convert_image_dtype(canvas, tf.float32)
    img = tf.image.resize(img, (64, 64))    
    img = np.array(img)
    img = np.expand_dims(img, axis=0)
    
    #plt.imshow(img)
    #print(img)

    # save the blank canvas to a file
    #canvas.save("unicode-text.png", "PNG")
    #canvas.show()
    return img

    
#CreateFontImage(draw_text, font)

def getFontImage(fontPath, imageNum):
    #fontFiles = getSpecificExtensionFiles(fontPath, ".ttf")
    
    #for i in range(1, imageNum):
        #fontidx = random.randrange(0, len(fontFiles) + 1)
    cho = random.randrange(0, 19)
    jung = random.randrange(0, 21)
    jong = random.randrange(0, 28)
    
    ja = label2ja[cho]
    mo = label2mo[jung]
    ba = label2ba[jong]

    char = unicode.join_jamos_char(ja, mo ,ba)
    #print(char)
    
    label1 = np.expand_dims(np.array(cho), axis=0)
    label2 = np.expand_dims(np.array(jung), axis=0)
    label3 = np.expand_dims(np.array(jong), axis=0)
    

    #yield CreateFontImage(char, fontFiles[fontidx])
    return CreateFontImage(char, fontPath), (label1, label2, label3)

In [8]:
def get_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    fontNum = len(fontFiles)
    
    os.system("shuf /root/Data/hangul/dataset/tranDataset.csv > /root/Data/hangul/dataset/shuffled_tranDataset.csv")
    csvFile = open("/root/Data/hangul/dataset/shuffled_tranDataset.csv", 'r', encoding='utf-8')
    #csvFile = open("/root/Data/hangul/dataset/test.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        yield img, (label1, label2, label3)
        
        fontidx = random.randrange(0, len(fontFiles))
        yield getFontImage(fontFiles[fontidx], 10)

        if (cnt > DATA_SIZE): break
        else                : cnt += 1

        
        
def get_valid_dataset_fromCsv():
    cnt = 0
    
    fontFiles = getSpecificExtensionFiles("/root/Data/font/clova-all", ".ttf")
    fontNum = len(fontFiles)

    os.system("shuf /root/Data/hangul/dataset/validation.csv > /root/Data/hangul/dataset/shuffled_validation.csv")    
    csvFile = open("/root/Data/hangul/dataset/shuffled_validation.csv", 'r', encoding='utf-8')
    reader = csv.reader(csvFile)
    
    for line in reader:
        imgFile = line[0]
        #print(imgFile)
        img = tf.io.read_file(imgFile)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = tf.image.resize(img, (64, 64))    
        img = np.array(img)
        img = np.expand_dims(img, axis=0)
        
        label1 = np.expand_dims(np.array(int(line[1])), axis=0)
        label2 = np.expand_dims(np.array(int(line[2])), axis=0)
        label3 = np.expand_dims(np.array(int(line[3])), axis=0)
        
        yield img, (label1, label2, label3)
        
        fontidx = random.randrange(0, len(fontFiles))
        yield getFontImage(fontFiles[fontidx], 10)

        if (cnt > VALID_DATA_SIZE): break
        else                : cnt += 1

In [9]:
dataset = tf.data.Dataset.from_generator(get_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )


validDtaset = tf.data.Dataset.from_generator(get_valid_dataset_fromCsv,
                                            output_signature=
                                            (
                                            tf.TensorSpec(shape = (None, 64, 64, 3), dtype =tf.float32, name = "posts"),
                                            (tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseCho2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJung2"),
                                            tf.TensorSpec(shape = (1,), dtype = tf.int64, name = "DenseJong2"))
                                            )
                                            #output_shapes=([19,21,28])
                                            )

2024-04-09 08:50:34.584769: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-09 08:50:34.584891: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-09 08:50:34.584929: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-09 08:50:34.585536: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-04-09 08:50:34.585580: I external/local_xla/xla/stream_executor

In [10]:
# 1
posts_input = Input(shape=(64,64,3), dtype='float32', name='posts')

# common layer 1

common = layers.Conv2D(filters= 32, kernel_size=(3,3), padding="same")(posts_input)
common = layers.BatchNormalization()(common)
common = layers.Activation('relu')(common)
common = layers.SpatialDropout2D(0.2)(common)
# common = layers.MaxPool2D(pool_size=2)(common)

common = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(common)
common = layers.BatchNormalization()(common)
common = layers.Activation('relu')(common)
common = layers.SpatialDropout2D(0.2)(common)
common = layers.MaxPool2D(pool_size=2)(common)

firstLayer = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(common)
firstLayer = layers.BatchNormalization()(firstLayer)
firstLayer = layers.Activation('relu')(firstLayer)

firstLayer = layers.Conv2D(filters= 128, kernel_size=(3,3), padding="same")(firstLayer)
firstLayer = layers.BatchNormalization()(firstLayer)
firstLayer = layers.Activation('relu')(firstLayer)
firstLayer = layers.SpatialDropout2D(0.2)(firstLayer)

firstLayer = layers.Conv2D(filters= 256, kernel_size=(3,3), padding="same")(firstLayer)
firstLayer = layers.BatchNormalization()(firstLayer)
firstLayer = layers.Activation('relu')(firstLayer)
firstLayer = layers.SpatialDropout2D(0.2)(firstLayer)

firstLayer = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(firstLayer)
firstLayer = layers.BatchNormalization()(firstLayer)
firstLayer = layers.Activation('relu')(firstLayer)
firstLayer = layers.SpatialDropout2D(0.2)(firstLayer)

SecondLayer = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(common)
SecondLayer = layers.BatchNormalization()(SecondLayer)
SecondLayer = layers.Activation('relu')(SecondLayer)

SecondLayer = layers.Conv2D(filters= 128, kernel_size=(3,3), paddibng="same")(SecondLayer)
SecondLayer = layers.BatchNormalization()(SecondLayer)
SecondLayer = layers.Activation('relu')(SecondLayer)
SecondLayer = layers.SpatialDropout2D(0.2)(SecondLayer)

SecondLayer = layers.Conv2D(filters= 256, kernel_size=(3,3), padding="same")(SecondLayer)
SecondLayer = layers.BatchNormalization()(SecondLayer)
SecondLayer = layers.Activation('relu')(SecondLayer)
SecondLayer = layers.SpatialDropout2D(0.2)(SecondLayer)

SecondLayer = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(SecondLayer)
SecondLayer = layers.BatchNormalization()(SecondLayer)
SecondLayer = layers.Activation('relu')(SecondLayer)
SecondLayer = layers.SpatialDropout2D(0.2)(SecondLayer)


ThirdLayer = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(common)
ThirdLayer = layers.BatchNormalization()(SecondLayer)
ThirdLayer = layers.Activation('relu')(SecondLayer)

ThirdLayer = layers.Conv2D(filters= 128, kernel_size=(3,3), padding="same")(ThirdLayer)
ThirdLayer = layers.BatchNormalization()(ThirdLayer)
ThirdLayer = layers.Activation('relu')(ThirdLayer)
ThirdLayer = layers.SpatialDropout2D(0.2)(ThirdLayer)

ThirdLayer = layers.Conv2D(filters= 256, kernel_size=(3,3), padding="same")(ThirdLayer)
ThirdLayer = layers.BatchNormalization()(ThirdLayer)
ThirdLayer = layers.Activation('relu')(ThirdLayer)
ThirdLayer = layers.SpatialDropout2D(0.2)(ThirdLayer)

ThirdLayer = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(ThirdLayer)
ThirdLayer = layers.BatchNormalization()(ThirdLayer)
ThirdLayer = layers.Activation('relu')(ThirdLayer)
ThirdLayer = layers.SpatialDropout2D(0.2)(ThirdLayer)


firstLayer = layers.Flatten()(firstLayer)
SecondLayer = layers.Flatten()(SecondLayer)
ThirdLayer = layers.Flatten()(ThirdLayer)
#SecondLayer = layers.Flatten()(SecondLayer)
#ThirdLayer = layers.Flatten()(ThirdLayer)

# DenseCho = layers.Dense(128, activation='relu', name='DenseCho1')(x)
# DenseJung = layers.Dense(128, activation='relu', name='DenseJung1')(x)
# DenseJong = layers.Dense(128, activation='relu', name='DenseJong1')(x)


DenseCho = layers.Dense(19, activation='softmax', name='DenseCho2')(firstLayer)
DenseJung = layers.Dense(21, activation='softmax', name='DenseJung2')(SecondLayer)
DenseJong = layers.Dense(28, activation='softmax', name='DenseJong2')(ThirdLayer)

losses = {
	#"DenseCho2": "categorical_crossentropy",
	"DenseCho2": "sparse_categorical_crossentropy",
	"DenseJung2": "sparse_categorical_crossentropy",
    "DenseJong2": "sparse_categorical_crossentropy"
}

model = Model(posts_input, [DenseCho, DenseJung, DenseJong])

model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

# Total params: 1,678,212 (6.40 MB)
#  Total params: 1,678,212 (6.40 MB)

In [ ]:
# 1
posts_input = Input(shape=(64,64,3), dtype='float32', name='posts')

# common layer 1

common = layers.Conv2D(filters= 32, kernel_size=(3,3), padding="same")(posts_input)
common = layers.BatchNormalization()(common)
common = layers.Activation('relu')(common)
# common layer 2

common = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(common)
common = layers.BatchNormalization()(common)
common = layers.Activation('relu')(common)
common = layers.SpatialDropout2D(0.2)(common)
common = layers.MaxPool2D(pool_size=2)(common)

common = layers.Conv2D(filters= 128, kernel_size=(3,3), padding="same")(common)
common = layers.BatchNormalization()(common)
common = layers.Activation('relu')(common)
common = layers.SpatialDropout2D(0.2)(common)
common = layers.MaxPool2D(pool_size=2)(common)

common = layers.Conv2D(filters= 256, kernel_size=(3,3), padding="same")(common)
common = layers.BatchNormalization()(common)
common = layers.Activation('relu')(common)
common = layers.SpatialDropout2D(0.2)(common)
common = layers.MaxPool2D(pool_size=2)(common)

common = layers.Conv2D(filters= 32, kernel_size=(3,3), padding="same")(posts_input)
common = layers.BatchNormalization()(common)
common = layers.Activation('relu')(common)
# common layer 2

lastCl = layers.Conv2D(filters= 64, kernel_size=(3,3), padding="same")(posts_input)
lastCl = layers.BatchNormalization()(lastCl)
lastCl = layers.Activation('relu')(lastCl)
lastCl = layers.SpatialDropout2D(0.2)(lastCl)
lastCl = layers.MaxPool2D(pool_size=2)(lastCl)

lastCl = layers.Conv2D(filters= 128, kernel_size=(3,3), padding="same")(lastCl)
lastCl = layers.BatchNormalization()(common)
lastCl = layers.Activation('relu')(common)
lastCl = layers.SpatialDropout2D(0.2)(lastCl)
lastCl = layers.MaxPool2D(pool_size=2)(lastCl)

lastCl = layers.Conv2D(filters= 256, kernel_size=(3,3), padding="same")(lastCl)
lastCl = layers.BatchNormalization()(lastCl)
lastCl = layers.Activation('relu')(lastCl)
lastCl = layers.SpatialDropout2D(0.2)(lastCl)
lastCl = layers.MaxPool2D(pool_size=2)(lastCl)


firstLayer = layers.Conv2D(filters = 128, kernel_size=(3,3), padding="same")(common)
firstLayer = layers.BatchNormalization()(firstLayer)
firstLayer = layers.Conv2D(filters = 64, kernel_size=(3,3), padding="same")(firstLayer)
firstLayer = layers.BatchNormalization()(firstLayer)
firstLayer = layers.AveragePooling2D(pool_size=2)(firstLayer)
firstLayer = layers.Conv2D(filters = 32, kernel_size=(3,3), padding="same")(firstLayer)
firstLayer = layers.BatchNormalization()(firstLayer)
firstLayer = layers.SpatialDropout2D(0.2)(firstLayer)

SecondLayer = layers.Conv2D(filters = 128, kernel_size=(3,3), padding="same")(lastCl)
SecondLayer = layers.BatchNormalization()(SecondLayer)
SecondLayer = layers.Conv2D(filters = 64, kernel_size=(3,3), padding="same")(SecondLayer)
SecondLayer = layers.BatchNormalization()(SecondLayer)
SecondLayer = layers.AveragePooling2D(pool_size=2)(SecondLayer)
SecondLayer = layers.Conv2D(filters = 32, kernel_size=(3,3), padding="same")(SecondLayer)
SecondLayer = layers.BatchNormalization()(SecondLayer)
SecondLayer = layers.SpatialDropout2D(0.2)(SecondLayer)

ThirdLayer = layers.Conv2D(filters = 128, kernel_size=(3,3), padding="same")(common)
ThirdLayer = layers.BatchNormalization()(ThirdLayer)
ThirdLayer = layers.Conv2D(filters = 64, kernel_size=(3,3), padding="same")(ThirdLayer)
ThirdLayer = layers.BatchNormalization()(ThirdLayer)
ThirdLayer = layers.AveragePooling2D(pool_size=2)(ThirdLayer)
ThirdLayer = layers.Conv2D(filters = 32, kernel_size=(3,3), padding="same")(ThirdLayer)
ThirdLayer = layers.BatchNormalization()(ThirdLayer)
ThirdLayer = layers.SpatialDropout2D(0.2)(ThirdLayer)

firstLayer = layers.Flatten()(firstLayer)
SecondLayer = layers.Flatten()(SecondLayer)
ThirdLayer = layers.Flatten()(ThirdLayer)

# DenseCho = layers.Dense(128, activation='relu', name='DenseCho1')(x)
# DenseJung = layers.Dense(128, activation='relu', name='DenseJung1')(x)
# DenseJong = layers.Dense(128, activation='relu', name='DenseJong1')(x)


DenseCho = layers.Dense(19, activation='softmax', name='DenseCho2')(firstLayer)
DenseJung = layers.Dense(21, activation='softmax', name='DenseJung2')(SecondLayer)
DenseJong = layers.Dense(28, activation='softmax', name='DenseJong2')(ThirdLayer)

losses = {
	#"DenseCho2": "categorical_crossentropy",
	"DenseCho2": "sparse_categorical_crossentropy",
	"DenseJung2": "sparse_categorical_crossentropy",
    "DenseJong2": "sparse_categorical_crossentropy"
}

model = Model(posts_input, [DenseCho, DenseJung, DenseJong])

model.compile(loss = 'sparse_categorical_crossentropy',optimizer='adam', 
               metrics=[['accuracy'], ['accuracy'], ['accuracy']])

# Total params: 1,678,212 (6.40 MB)

In [11]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ posts (InputLayer)  │ (None, 64, 64, 3) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        896 │ posts[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d   │ (None, 64, 64,    │          0 │ activation[0][0]  │
│ (SpatialDropout2D)  │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 64, 64,    │     18,496 │ spatial_dropout2… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d_1 │ (None, 64, 64,    │          0 │ activation_1[0][… │
│ (SpatialDropout2D)  │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ spatial_dropout2… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 32, 32,    │     36,928 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 32, 32,    │     73,856 │ activation_6[0][… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d_5 │ (None, 32, 32,    │          0 │ activation_7[0][… │
│ (SpatialDropout2D)  │ 128)              │            │                 

 Total params: 6,105,668 (23.29 MB)

 Trainable params: 6,102,532 (23.28 MB)

 Non-trainable params: 3,136 (12.25 KB)

In [12]:
save_dir = "/root/Data/hangul/weights"
checkPoint_path = save_dir + "/handwriteModeling2_1.weights.h5"

#from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
# 3번 반복내에 validation loss가 줄어들지 않으면 learning rate를 0.2 감소
#lr_cb = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, mode='min', verbose=1)
# 5번 반복내에 validation loss가 줄어들지 않으면 강제종료
#st_cb = EarlyStopping(monitor='val_loss', patience=5, mode='min', verbose=1)

cp_callback = keras.callbacks.ModelCheckpoint(filepath = checkPoint_path, save_weights_only=True, save_best_only=True, monitor = 'loss')

#model.fit(dataset, validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback])
model.fit(dataset, batch_size = 16, epochs = 100, callbacks=[cp_callback], validation_data= validDtaset)
#model.train_on_batch(dataset)

Epoch 1/100


I0000 00:00:1712620286.445388   19215 service.cc:145] XLA service 0x7f3768001ea0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1712620286.445436   19215 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-04-09 08:51:26.648760: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-04-09 08:51:27.429542: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


      1/Unknown 26s 26s/step - DenseCho2_accuracy: 0.0000e+00 - DenseJong2_accuracy: 0.0000e+00 - DenseJung2_accuracy: 0.0000e+00 - loss: 13.8013

I0000 00:00:1712620303.410323   19215 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  60001/Unknown 778s 13ms/step - DenseCho2_accuracy: 0.4009 - DenseJong2_accuracy: 0.3722 - DenseJung2_accuracy: 0.3635 - loss: 8.0260

...

60003/60004 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - DenseCho2_accuracy: 0.9141 - DenseJong2_accuracy: 0.8944 - DenseJung2_accuracy: 0.8834 - loss: 1.0740

2024-04-09 12:35:19.461771: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-04-09 12:35:19.461863: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_8]]


60004/60004 ━━━━━━━━━━━━━━━━━━━━ 771s 13ms/step - DenseCho2_accuracy: 0.9141 - DenseJong2_accuracy: 0.8944 - DenseJung2_accuracy: 0.8834 - loss: 1.0740 - val_DenseCho2_accuracy: 0.8322 - val_DenseJong2_accuracy: 0.8326 - val_DenseJung2_accuracy: 0.8278 - val_loss: 2.4469
Epoch 18/100


2024-04-09 12:36:07.452361: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-04-09 12:36:07.452412: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-04-09 12:36:07.452442: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 3235305867090892202
2024-04-09 12:36:07.452475: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10468017040788610711


13723/60004 ━━━━━━━━━━━━━━━━━━━━ 9:25 12ms/step - DenseCho2_accuracy: 0.9083 - DenseJong2_accuracy: 0.8898 - DenseJung2_accuracy: 0.8833 - loss: 1.1201

In [11]:
save_dir = "/root/Data/hangul/weights"
checkPoint_path = save_dir + "/handwriteModeling1_6.weights.h5"

#from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
# 3번 반복내에 validation loss가 줄어들지 않으면 learning rate를 0.2 감소
#lr_cb = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, mode='min', verbose=1)
# 5번 반복내에 validation loss가 줄어들지 않으면 강제종료
#st_cb = EarlyStopping(monitor='val_loss', patience=5, mode='min', verbose=1)

cp_callback = keras.callbacks.ModelCheckpoint(filepath = checkPoint_path, save_weights_only=True, save_best_only=True, monitor = 'loss')

#model.fit(dataset, validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback])
model.fit(dataset, batch_size = 16, epochs = 100, callbacks=[cp_callback], validation_data= validDtaset)
#model.train_on_batch(dataset)

Epoch 1/100


I0000 00:00:1712187372.097148   13449 service.cc:145] XLA service 0x7f5c9c003f50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1712187372.097202   13449 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-04-04 08:36:12.318249: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-04-04 08:36:13.185801: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


      3/Unknown 25s 36ms/step - DenseCho2_accuracy: 0.0000e+00 - DenseJong2_accuracy: 0.0000e+00 - DenseJung2_accuracy: 0.2778 - loss: 11.7140  

I0000 00:00:1712187386.668964   13449 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  60001/Unknown 515s 8ms/step - DenseCho2_accuracy: 0.5182 - DenseJong2_accuracy: 0.5829 - DenseJung2_accuracy: 0.4668 - loss: 5.7315

...

60004/60004 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - DenseCho2_accuracy: 0.9497 - DenseJong2_accuracy: 0.9463 - DenseJung2_accuracy: 0.9430 - loss: 0.5819

2024-04-04 15:43:32.204225: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-04-04 15:43:32.204262: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_8]]
2024-04-04 15:43:32.204273: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 8459907106845558289
2024-04-04 15:43:32.204279: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 6885617732219031438


60004/60004 ━━━━━━━━━━━━━━━━━━━━ 521s 9ms/step - DenseCho2_accuracy: 0.9497 - DenseJong2_accuracy: 0.9463 - DenseJung2_accuracy: 0.9430 - loss: 0.5819 - val_DenseCho2_accuracy: 0.9269 - val_DenseJong2_accuracy: 0.9141 - val_DenseJung2_accuracy: 0.9168 - val_loss: 1.0832
Epoch 50/100
   13/60004 ━━━━━━━━━━━━━━━━━━━━ 9:23 9ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 0.9877 - DenseJung2_accuracy: 0.8200 - loss: 0.2338  

2024-04-04 15:44:12.706151: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-04-04 15:44:12.706190: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_4]]
2024-04-04 15:44:12.706202: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 8459907106845558289
2024-04-04 15:44:12.706208: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1615523814258982773
2024-04-04 15:44:12.706213: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10905273783146134026
2024-04-04 15:44:12.706236: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 6885617732219031438


30176/60004 ━━━━━━━━━━━━━━━━━━━━ 4:04 8ms/step - DenseCho2_accuracy: 0.9484 - DenseJong2_accuracy: 0.9452 - DenseJung2_accuracy: 0.9415 - loss: 0.5873

In [11]:
save_dir = "/root/Data/hangul/weights"
checkPoint_path = save_dir + "/handwriteModeling1_5.weights.h5"

cp_callback = keras.callbacks.ModelCheckpoint(filepath = checkPoint_path, save_weights_only=True, save_best_only=True, monitor = 'loss')


#model.fit(dataset, validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback])
model.fit(dataset, batch_size = 16, epochs = 100, callbacks=[cp_callback], validation_data= validDtaset)
#model.train_on_batch(dataset)

Epoch 1/100


I0000 00:00:1712129105.631619  162032 service.cc:145] XLA service 0x7f5330002530 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1712129105.631658  162032 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-04-03 16:25:05.809203: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-04-03 16:25:06.512980: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


      1/Unknown 23s 23s/step - DenseCho2_accuracy: 0.0000e+00 - DenseJong2_accuracy: 0.0000e+00 - DenseJung2_accuracy: 0.0000e+00 - loss: 12.9025

I0000 00:00:1712129118.861116  162032 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  60004/Unknown 1258s 21ms/step - DenseCho2_accuracy: 0.5200 - DenseJong2_accuracy: 0.5639 - DenseJung2_accuracy: 0.4812 - loss: 5.7515

...

60004/60004 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - DenseCho2_accuracy: 0.9257 - DenseJong2_accuracy: 0.9281 - DenseJung2_accuracy: 0.9122 - loss: 0.8556

2024-04-03 17:51:21.268513: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-04-03 17:51:21.268565: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 502333051309010726
2024-04-03 17:51:21.268592: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2024-04-03 17:51:21.268624: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 2003008956880002767


60004/60004 ━━━━━━━━━━━━━━━━━━━━ 1332s 22ms/step - DenseCho2_accuracy: 0.9257 - DenseJong2_accuracy: 0.9281 - DenseJung2_accuracy: 0.9122 - loss: 0.8556 - val_DenseCho2_accuracy: 0.8635 - val_DenseJong2_accuracy: 0.8935 - val_DenseJung2_accuracy: 0.8811 - val_loss: 1.6749
Epoch 5/100


2024-04-03 17:52:19.445910: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-04-03 17:52:19.445949: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_8]]
2024-04-03 17:52:19.445960: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10845780414974048445
2024-04-03 17:52:19.445964: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 11956488142041423543
2024-04-03 17:52:19.445968: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 2003008956880002767
2024-04-03 17:52:19.445993: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 502333051309010726


38062/60004 ━━━━━━━━━━━━━━━━━━━━ 7:48 21ms/step - DenseCho2_accuracy: 0.9315 - DenseJong2_accuracy: 0.9294 - DenseJung2_accuracy: 0.9193 - loss: 0.8064

In [11]:
save_dir = "/root/Data/hangul/weights"
checkPoint_path = save_dir + "/handwriteModeling1_3.weights.h5"

cp_callback = keras.callbacks.ModelCheckpoint(filepath = checkPoint_path, save_weights_only=True, save_best_only=True, monitor = 'loss')


#model.fit(dataset, validDtaset, batch_size = 16, epochs = 100, callbacks=[cp_callback])
model.fit(dataset, batch_size = 16, epochs = 100, callbacks=[cp_callback], validation_data= validDtaset)
#model.train_on_batch(dataset)

Epoch 1/100


I0000 00:00:1712029132.858055   91512 service.cc:145] XLA service 0x7f7988003fe0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1712029132.858118   91512 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce GTX 1650, Compute Capability 7.5
2024-04-02 12:38:52.996905: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-04-02 12:38:53.544848: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


      7/Unknown 17s 22ms/step - DenseCho2_accuracy: 0.2276 - DenseJong2_accuracy: 0.0728 - DenseJung2_accuracy: 0.3704 - loss: 16.8459     

I0000 00:00:1712029142.037052   91512 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  60000/Unknown 492s 8ms/step - DenseCho2_accuracy: 0.4811 - DenseJong2_accuracy: 0.5559 - DenseJung2_accuracy: 0.4659 - loss: 8.1908

...

60002/60004 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - DenseCho2_accuracy: 0.9353 - DenseJong2_accuracy: 0.9367 - DenseJung2_accuracy: 0.9244 - loss: 0.8096

2024-04-02 16:28:56.025241: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-04-02 16:28:56.025320: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-04-02 16:28:56.025334: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 2239930974207642251
2024-04-02 16:28:56.025342: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 4586870833631338328


60004/60004 ━━━━━━━━━━━━━━━━━━━━ 516s 9ms/step - DenseCho2_accuracy: 0.9353 - DenseJong2_accuracy: 0.9367 - DenseJung2_accuracy: 0.9244 - loss: 0.8096 - val_DenseCho2_accuracy: 0.8972 - val_DenseJong2_accuracy: 0.8866 - val_DenseJung2_accuracy: 0.8605 - val_loss: 1.5928
Epoch 28/100
    1/60004 ━━━━━━━━━━━━━━━━━━━━ 1:03:22 63ms/step - DenseCho2_accuracy: 1.0000 - DenseJong2_accuracy: 1.0000 - DenseJung2_accuracy: 1.0000 - loss: 4.8513e-04

2024-04-02 16:29:35.461100: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-04-02 16:29:35.461146: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]


31297/60004 ━━━━━━━━━━━━━━━━━━━━ 3:44 8ms/step - DenseCho2_accuracy: 0.9367 - DenseJong2_accuracy: 0.9361 - DenseJung2_accuracy: 0.9210 - loss: 0.8089

In [12]:
model.load_weights("/root/Data/hangul/weights/handwriteModeling1_7.weights.h5")

/root/anaconda3/envs/Anaconda_tensor/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:396: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 134 variables. 
  trackable.load_own_variables(weights_store.get(inner_path))


In [66]:
#img = tf.io.read_file("/root/Data/hangul_handwrite_val/image/test/26.jpg")
#img = tf.io.read_file("/root/venv/Anaconda/Anaconda_Tensor/TrainData/image/20.jpg")
#img = tf.io.read_file("/root/Data/hangul_handwrite_val/image//31.jpg")
#img = tf.io.read_file("E:\\unzipData\\Training\\image_Training_handwrite\\1.letter\\040\\04030003097")
img = tf.io.read_file("/root/Data/hangul_handwrite_val/image/test/young.jpg")
img = tf.image.decode_jpeg(img, channels=3)
img = tf.image.convert_image_dtype(img, tf.float32)
img = tf.image.resize(img, (64, 64))

print(img.shape)

img = np.array(img)

img = np.expand_dims(img, axis=0)

#print(img.shape)

ch, ju, jo = model.predict(img)

ch = ch.argmax()
ju = ju.argmax()
jo = jo.argmax()

ja = label2ja[ch]
mo = label2mo[ju]
ba = label2ba[jo]

char = unicode.join_jamos_char(ja, mo ,ba)
print(char)

(64, 64, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
명


In [67]:
#model.save("./testModel.h5")
model.save('./handwriteModeling1_7.keras')